# Topic modelling 

This workbook implements topic modelling - using the optimal approach from 1_0_2_topic_modelling_practise.ipynb. 

In [1]:
import pandas as pd
import numpy as np
import re
import string 
from datasets import Dataset

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster import hierarchy as sch

from transformers import AutoModelForMaskedLM, pipeline, AutoTokenizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

import nltk 
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

from gensim import corpora
from gensim.models import LdaModel

from hdbscan import HDBSCAN
from umap import UMAP

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read the remote database of comments 

In [2]:
cs = CommentsSaver(env='dev')
df = cs.read_all()
print('df shape:', df.shape)

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.
df shape: (30393, 12)


In [3]:
df.columns

Index(['id', 'council', 'comment_id', 'application_id', 'address', 'stance',
       'date', 'comment_text', 'add_date', 'lat', 'lon',
       'cleaned_comment_text'],
      dtype='object')

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = model.tokenizer

### Pre-process the data

1. Remove non-ASCII characters 

Just because otherwise these are a bit of a pain

2. Remove place names 

I don't want the topics identified to relate to the place names of specific applications (i.e. Durning Hall or Forest Gate) - as I want the topics to be generalised themes, common across applications - hence I remove all place names. This uses Named Entity Recognition (NER), I intially tried using the out of the box bog-standard model, but it wasn't able to recognise more specific British place names. Instead I use the "cjber/reddit-ner-place_names" - which has specifically been trained to recognise these sorts of place names. 

```
place_ner_pipeline = pipeline(
    task="ner",
    model="cjber/reddit-ner-place_names",
    tokenizer="cjber/reddit-ner-place_names",
    aggregation_strategy="first",
)
```

3. Remove peoples names 

I don't want the topics identified to relate to individuals. Also, this helps anonymise the data. 

```
people_ner_pipeline = pipeline(
            task="ner",
            model="dslim/bert-base-NER",
            tokenizer="dslim/bert-base-NER",
            aggregation_strategy="first"
        )
```

NOTE: I leave in stop words and the like --- I'm using a sentence transformer ('all-MiniLM-L6-v2') so I want to preserve sentence structure. 

In [5]:
df['cleaned_comment_text']

0        There is a pre-existing extension to the rear ...
1        Dear ,\n\nCould you help me please I understan...
2        This going going g to be great for , I use to ...
3        It has been 7 years now since Women's Pioneer ...
4        I support the purpose of the re-development bu...
                               ...                        
30388    Building a 15 storey block will impact unfavou...
30389    While the aims of the project are well meant, ...
30390    A 15-storey building is just too high for our ...
30391    Whilst 64 has undoubtedly development potentia...
30392    I broadly support the social aims of the proje...
Name: cleaned_comment_text, Length: 30393, dtype: object

In [6]:
nlp_tasks = NLP_Tasks()
# split text on newlines, this function preserves the metadata by exploding the dataframe
train_df_split = nlp_tasks.split_text_on_newline(df=df, column='cleaned_comment_text')

Device set to use mps:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [7]:
len(train_df_split)

151373

In [8]:
cleaned_text = train_df_split['cleaned_comment_text'].tolist()

### Repeat BertTopic modelling with geographic place names removed 

This uses the same fine-tuned huggingface transformer model fine-tuned to have domain specific knowledge from earlier. But, I have done some additional data processing, and I specify some of the models hyper parameters. 

In [9]:
# this controls the seed - allowing for reproducible maps 
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=43)

# this controls the topic parameters
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [10]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(cleaned_text, show_progress_bar=True)

Batches: 100%|██████████| 4731/4731 [09:49<00:00,  8.03it/s] 


In [ ]:
topic_model_1 = BERTopic(hdbscan_model=hdbscan_model, umap_model=umap_model, verbose=True, calculate_probabilities=True)
topics, probs = topic_model_1.fit_transform(cleaned_text, embeddings)

2025-05-30 14:34:34,399 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-30 14:37:47,083 - BERTopic - Dimensionality - Completed ✓
2025-05-30 14:37:47,089 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been us

In [ ]:
# Display the topics found
topic_model_1.get_topic_info()

Save the model using PyTorch 

In [21]:
# Save the topic model 
bertopic_model_dir = '/Users/bea/Documents/AI4CI/projects/comment_summariser/comment_crunch/outputs/bertopic_fine_tuning'
topic_model_no_place.save(bertopic_model_dir, serialization="pytorch", save_ctfidf=True, save_embedding_model=model)

In [22]:
# Load the topic model
loaded_topic_model_no_place = BERTopic.load(bertopic_model_dir, embedding_model=model)

Some further analysis of the topic information

In [23]:
def print_topic_info(n):
    freq = topic_model_no_place.get_topic_freq(n)
    print(f'Frequency of the topic in the training dataset: {freq}\n')
    print(f'Representative documents of the topic:\n')
    print(f'{topic_model_no_place.get_representative_docs(n)[0]}')
    # for rep in topic_model_no_place.get_representative_docs(n):
    #     print(rep)

In [24]:
print_topic_info(1)

Frequency of the topic in the training dataset: 197

Representative documents of the topic:

Loss of amenity. Owners at were sold on the significant green space that would form part of the development. Approved planning applications for the development previously emphasised the amount of green space. now intend to take this away, having misled leaseholders into believing it would remain green space.


In [25]:
# `topic_distr` contains the distribution of topics in each document
topic_distr, _ = topic_model_no_place.approximate_distribution(cleaned_place_text, window=8, stride=4)

100%|██████████| 3/3 [00:00<00:00, 20.95it/s]


In [26]:
topic_model_no_place.visualize_topics()

In [27]:
topic_model_no_place.visualize_heatmap()

In [28]:
# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
hierarchical_topics = topic_model_no_place.hierarchical_topics(cleaned_place_text, linkage_function=linkage_function)

100%|██████████| 45/45 [00:00<00:00, 693.81it/s]


In [29]:
topic_model_no_place.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

### Out of set model performance 

Above I looked at identifying all topics for my training dataset. So what if I now try to assign these topics to a smaller subset - here all the comments relating to a single planning application. 

The planning applications considered:
* Ealing 223203FUL

In [30]:
cs = CommentsSaver(env='prod')
df = cs.read_all()
df_ealing = df[df['application_id']=='223203FUL']

Connecting to the ai4ci-db database...
Successfully connected to ai4ci-db.


In [31]:
df_ealing.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
7965,9457,Ealing,223203FUL_112,223203FUL,34 lexden road london w3 9nz London W3 9NZ W3 9NZ,Objects,2022-08-25,Taking away trees .enough of highrise flats in...,2025-03-21,NaN,NaN
7975,9355,Ealing,223203FUL_21,223203FUL,10 Baldwyn Gardens London W3 6HH W3 6HH,Objects,2022-09-16,Yet another attempt by developers to push thro...,2025-03-21,NaN,NaN
8029,9332,Ealing,223203FUL_16,223203FUL,35 Cumberland Park London w3 6sx w3 6sx,Objects,2022-09-16,The proposals will not only be an eyesore and ...,2025-03-21,NaN,NaN
8032,9371,Ealing,223203FUL_35,223203FUL,34 Derwentwater Road LONDON W3 6DF W3 6DF,Objects,2022-09-10,The centre of Acton is overcrowded. The new bl...,2025-03-21,NaN,NaN
8034,9336,Ealing,223203FUL_1,223203FUL,177 Horn Lane Flat 2 London W3 6PW W3 6PW,Neutral,2022-10-09,In the shrubs next to the road you can find sh...,2025-03-21,NaN,NaN


In [32]:
print(f'Number of comments for application 223203FUL in Ealing: {len(df_ealing)}')

Number of comments for application 223203FUL in Ealing: 160


In [33]:
# remove place names, numbers and non-ASCII characters
df_ealing = nlp.remove_place_names(df=df_ealing, column='comment_text')
df_ealing = nlp.remove_numbers(df=df_ealing, column='cleaned_comment_text')
df_ealing = nlp.remove_non_ascii(df=df_ealing, column='cleaned_comment_text')

# # split text on newlines
df_ealing_split = nlp.split_text_on_newline(df=df_ealing, column='cleaned_comment_text')

In [34]:
cleaned_ealing_text = df_ealing_split['cleaned_comment_text'].tolist()

In [35]:
# Apply previous topic model to the new dataset of comments from the one application in Ealing 
topics_subset, probs_subset = topic_model_no_place.transform(cleaned_ealing_text)

Batches: 100%|██████████| 16/16 [00:07<00:00,  2.19it/s]
2025-05-27 10:30:35,186 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-27 10:30:38,138 - BERTopic - Dimensionality - Completed ✓
2025-05-27 10:30:38,138 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-27 10:30:38,149 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-05-27 10:30:38,192 - BERTopic - Probabilities - Completed ✓
2025-05-27 10:30:38,192 - BERTopic - Cluster - Completed ✓


In [36]:
# # Get topic labels from BERTopic
# topic_info = topic_model_no_place.get_topic_info()
# topic_labels = [f"{row['Topic']}: {row['Representation'][:30]}" for _, row in topic_info.iterrows()]

# # Create figure and axis
# fig, ax = plt.subplots(figsize=(12, 6))

# # Display heatmap using imshow
# cax = ax.imshow(probs_array, cmap="Blues", aspect="auto")

# # Add colorbar
# fig.colorbar(cax, ax=ax, label="Probability")

# # Labels
# ax.set_xlabel("Topic")
# ax.set_ylabel("Text Entry")
# ax.set_title("Topic Probability Heatmap for subset_text")

# # Set x-axis tick labels
# ax.set_xticks(np.arange(len(topic_labels)))  # Set tick positions
# ax.set_xticklabels(topic_labels, rotation=90, ha="right")  # Set tick labels

# # Show plot
# plt.show()


In [37]:
# # Identify topics with a probability > 0.5 in at least one entry
# selected_topics = np.where(probs_array > 0.5)[1]  # Get topic indices
# selected_topics = np.unique(selected_topics)  # Remove duplicates

# # Find the main topic (topic with highest probability) for each entry
# main_topics = np.argmax(probs_array, axis=1)  # Index of highest probability topic for each entry

# # Count occurrences of each main topic
# topic_counts = pd.Series(main_topics).value_counts().reset_index()
# topic_counts.columns = ["Topic", "Count"]

# # Get topic representations from BERTopic
# topic_info = topic_model_no_place.get_topic_info()

# # Filter topic_info to only include selected topics
# filtered_topics = topic_info[topic_info["Topic"].isin(selected_topics)][["Topic", "Representation"]]

# # Merge with topic counts
# final_topics = filtered_topics.merge(topic_counts, on="Topic", how="left").fillna(0)

# # Display the result
# print(final_topics)

Identify the frequency and probability of assignment across a range of different topics. 

In [38]:
# Convert probabilities to NumPy array
probs_array = np.array(probs_subset)

# Identify topics with a probability > 0.5 in at least one entry
selected_topics = np.where(probs_array > 0.5)[1]  
selected_topics = np.unique(selected_topics)  

# Find the main topic for each document
main_topics = np.argmax(probs_array, axis=1)

# Count occurrences of each main topic
topic_counts = pd.Series(main_topics).value_counts().reset_index()
topic_counts.columns = ["Topic", "Count"]

# Get topic representations from BERTopic (full version)
topic_info = topic_model_no_place.get_topic_info()
filtered_topics = topic_info[topic_info["Topic"].isin(selected_topics)][["Topic", "Representation"]]

# Merge with topic counts
final_topics = filtered_topics.merge(topic_counts, on="Topic", how="left").fillna(0)

# Get representative documents: Take 3 example documents for each topic
representative_docs = {}
for topic in selected_topics:
    doc_indices = np.where(main_topics == topic)[0]  # Get indices of docs assigned to this topic
    top_docs = [cleaned_ealing_text[i] for i in doc_indices[:3]]  # Get up to 3 docs
    representative_docs[topic] = top_docs

# Add representative documents to the dataframe
final_topics["Representative Docs"] = final_topics["Topic"].map(representative_docs)

# Display full data
pd.set_option("display.max_colwidth", None)  # Ensure full text is shown
final_topics

,Topic,Representation,Count,Representative Docs
0,0,"[community, space, for, to, that, hall, and, in, the, durning]",9,"[Transformation of footpath to a road - removing local community asset, Transformation of footpath to a road - removing local community asset, Transformation of footpath to a road - removing local community asset]"
1,1,"[green, space, was, this, would, the, of, in, development, as]",42,"[. Affecting ecology - Loss of mature trees and biodiversity. I note that the Bat report suggests that 'habitats supported on the site at provide no obvious roosting opportunities for bats'. I know this not to be true as our garden which backs onto the site has regular bat activity during twilight hours in the summer months. There are some beautiful trees on the site which not only aid the biodiversity of the area but are beneficial for the well bing of the residents. I believe that the council should be protecting these mature trees not taking them down., . Open space - the proposed increase in in residential properties, with resulting lack of open space is also of particular concern. In a post COVID world, planners should be creating developments with more open space not less., . Affecting ecology - Loss of mature trees and biodiversity. I note that the Bat report suggests that 'habitats supported on the site at provide no obvious roosting opportunities for bats'. I know this not to be true as our garden which backs onto the site has regular bat activity during twilight hours in the summer months. There are some beautiful trees on the site which not only aid the biodiversity of the area but are beneficial for the well bing of the residents. I believe that the council should be protecting these mature trees not taking them down.]"
2,2,"[school, nursery, next, children, construction, primary, and, noise, dust, building]",15,"[. Out of keeping with surrounding area, overbearing., I take my children swimming and to meet friends in , from south, and am disappointed to see more mature trees being removed to make space for yet another high rise building, which is out of keeping with local surroundings. Please reconsider this proposal. Thank you., . Out of keeping with surrounding area, overbearing.]"
3,3,"[sunlight, daylight, privacy, building, light, north, the, will, living, of]",44,"[This is outrageous. How on earth can you think this is fitting into the local area?, Loss of daylight, sunlight and privacy of neighbours, creates a loss of daylight, sunlight and privacy of neighbours]"
4,4,"[height, high, street, the, area, of, keeping, design, is, buildings]",58,"[Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area, and only add more pressure to the already over-stretched infrastructure and services of . On top of that, the electrical power requirements of these developments will severely strain our services to breaking point. Fully object., . Over development and out of keeping - the plans as presented are completely out of keeping with the local area. While and are storeys, they are an anomaly in . While some sensitive development of the site and improvements for existing residents is surely welcome, buildings of this height and mass will dominate the local skyline, reduce light and amenity for neighbours and put a strain on existing local facilities. The council should be looking to build to a much lower height to better integrate the site with the surrounding area., . Over development and out of keeping - the plans as presented are completely out of keeping with the local area. While and are storeys, they are an anomaly in . While some sensitive development of the site and improvements for existing residents is surely welcome, buildings of this height and mass will dominate the local skyline, reduce light and amenity for neighbours and put a strain on existing local facilities. The council should be looking to build to a m

### See if there are comments relating to specific topics

From reading the committee report for Ealing application 223203FUL I know there should be comments specifcally relating to both the scale and density of the development. Let's see if I can force identification of these topics. 

First I try using a simple regex match. 

In [39]:
# Define keyword lists for each topic
scale_keywords = ["large-scale", "expansion", "size of development", "massive project", "extent", "scale"]
density_keywords = ["density", "overcrowding", "high-rise", "too many units", "population increase", "compact"]

# Search subset_text for keyword matches
scale_texts = [text for text in cleaned_ealing_text if any(word in text.lower() for word in scale_keywords)]
density_texts = [text for text in cleaned_ealing_text if any(word in text.lower() for word in density_keywords)]

# Print results
print(f"Texts related to 'Scale of the development': {len(scale_texts)}")
print(scale_texts[:5])  # Show first 5 examples

print(f"\nTexts related to 'Density of the development': {len(density_texts)}")
print(density_texts[:5])  # Show first 5 examples

Texts related to 'Scale of the development': 12
['Design/ Massing/ Scale:', "Although I am not totally against the development of additional buildings on the site, building 'A' is not in keeping. The overall massing is far greater than that of the existing towers. In addition, building 'A' sits directly adjacent the highway/  storey homes, something that does not relate to the exiting character. The building should be set back considerably further from the main road. From a townscape perspective, the overall massing and scale (footprint, height) should be reduced by at least % to make it anywhere near acceptable.", 'This scale of this project is not in keeping with the local landscape and minimal information about its management and maintenance has been provided.', 'Utility supply will be put under further pressure because of the massive scale of this planned development, especially electricity, water and sewage handing.', 'The proposed Block A will make this blight considerably worse,

In [40]:
# Step 1: Transform scale & density texts to get topic assignments
scale_topics, scale_probs = topic_model_no_place.transform(scale_texts)
density_topics, density_probs = topic_model_no_place.transform(density_texts)

# Step 2: Identify main topic (highest probability) for each text
scale_main_topics = np.argmax(scale_probs, axis=1)
density_main_topics = np.argmax(density_probs, axis=1)

# Step 3: Count occurrences of each topic
scale_topic_counts = pd.Series(scale_main_topics).value_counts().reset_index()
scale_topic_counts.columns = ["Topic", "Scale_Count"]

density_topic_counts = pd.Series(density_main_topics).value_counts().reset_index()
density_topic_counts.columns = ["Topic", "Density_Count"]

# Step 4: Get topic representations from BERTopic
topic_info = topic_model_no_place.get_topic_info()

# Step 5: Merge counts with topic representations
scale_topic_info = topic_info.merge(scale_topic_counts, on="Topic", how="inner")
density_topic_info = topic_info.merge(density_topic_counts, on="Topic", how="inner")

# Step 6: Display results
print("\nTopics most related to 'Scale of the development':")
print(scale_topic_info[["Representation", "Scale_Count"]].sort_values(by="Scale_Count", ascending=False))

print("\nTopics most related to 'Density of the development':")
print(density_topic_info[["Representation", "Density_Count"]].sort_values(by="Density_Count", ascending=False))


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]
2025-05-27 10:30:39,721 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-27 10:30:39,757 - BERTopic - Dimensionality - Completed ✓
2025-05-27 10:30:39,757 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-27 10:30:39,758 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-05-27 10:30:39,762 - BERTopic - Probabilities - Completed ✓
2025-05-27 10:30:39,762 - BERTopic - Cluster - Completed ✓
Batches: 100%|██████████| 2/2 [00:02<00:00,  1.00s/it]
2025-05-27 10:30:41,765 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-27 10:30:41,880 - BERTopic - Dimensionality - Completed ✓
2025-05-27 10:30:41,880 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-27 10:30:41,882 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-05-27 10:30:41,888 - BERTopic - Proba


Topics most related to 'Scale of the development':
                                                                         Representation  \
2                 [height, high, street, the, area, of, keeping, design, is, buildings]   
0                        [community, space, for, to, that, hall, and, in, the, durning]   
1                        [green, space, was, this, would, the, of, in, development, as]   
3         [dlr, already, amenities, services, bus, service, tfl, gym, with, facilities]   
4         [construction, this, cocp, ccd, code, built, disagree, please, practice, don]   
5           [pipes, system, repairs, diverted, angles, sewer, right, are, that, sewage]   
6  [compensate, conditional, light, owners, must, agreement, low, affected, made, rise]   

   Scale_Count  
2            6  
0            1  
1            1  
3            1  
4            1  
5            1  
6            1  

Topics most related to 'Density of the development':
                             

### Trying to seed topics for BERTopic 

Now try an alternative approach where I seed topics in BERTopic. This works by predefining a subset of topics for the model, and then these are asisgned higher model weights, encouraging the model to identify these as topic clusters, whilst continuing to cluster on additional topics. 

In [41]:
seed_topic_list = [
    ["large-scale", "expansion", "size of development", "massive project", "extent", "scale"],
    ["density", "overcrowding", "high-rise", "too many units", "population increase", "compact"]
]

max_len = max([len(topic) for topic in seed_topic_list])
padded_list = [topic + [""]*(max_len-len(topic)) for topic in seed_topic_list]

In [42]:
# Define the model. This uses the same HDBSCAN and UMAP parameters as before.
topic_model_seed = BERTopic(embedding_model=model, hdbscan_model=hdbscan_model, umap_model=umap_model, verbose=True, calculate_probabilities=True, seed_topic_list=padded_list)

topics, probs = topic_model_seed.fit_transform(cleaned_place_text)

topic_model_seed.get_topic_info()[:10]

2025-05-27 10:30:41,914 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 78/78 [00:12<00:00,  6.38it/s]
2025-05-27 10:30:54,177 - BERTopic - Embedding - Completed ✓
2025-05-27 10:30:54,177 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]
2025-05-27 10:30:55,517 - BERTopic - Guided - Completed ✓
2025-05-27 10:30:55,518 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-27 10:31:03,242 - BERTopic - Dimensionality - Completed ✓
2025-05-27 10:31:03,242 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-27 10:31:03,417 - BERTopic - Cluster - Completed ✓
2025-05-27 10:31:03,419 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-05-27 10:31:03,478 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,571,-1_and_the_to_this,"[and, the, to, this, of, in, is, for, area, development]","[Finally, I am disappointed and concerned by the lack of visibility in the consultation process for this development. There are as far as I am aware no public displays of the plans. There has been no flyering/leafleting in the local area. These plans are for a development that is likely to have a massive impact on yet the developers seem to have made little effort to publicise their plans. It looks very much like an attempt to push this through under the radar though I hope very much that that isn't the case and that these comments will be considered and taken on board., Finally, I am disappointed and concerned by the lack of visibility in the consultation process for this development. There are as far as I am aware no public displays of the plans. There has been no flyering/leafleting in the local area. These plans are for a development that is likely to have a massive impact on yet the developers seem to have made little effort to publicise their plans. It looks very much like an attempt to push this through under the radar though I hope very much that that isn't the case and that these comments will be considered and taken on board., ) It is planned to access the development through and . This will have a substantial impact for residents during construction and will increase traffic flow subsequently as it will be the main access for the proposed new buildings. This will create noise pollution, congestion and a general loss of amenity to residents and is likely to cause additional parking issues on the already crowded estate roads.]"
1,0,305,0_community_space_for_to,"[community, space, for, to, that, hall, and, we, durning, in]","[As to information on alternative space, if we are to be ousted against our will then of course we would be happy to be provided with this, but that does not mean we are happy about having to leave Durning Hall, whether temporarily or permanently. In fact, Aston-Mansfield has not provided us with any such information since Roxana's e-mail and call. Quite some time before, they suggested to one of our scout leaders that during the building works we could use Forest Gate Community School, Forest Lane Lodge, The Gate and Vicarage Lane Community Centre (all of which are unsuitable for our needs), Field Community Centre (which has been privately leased and unavailable for years) and Upton Community Centre (which was demolished years ago). Obviously, these were no use to us, and there has been no follow-up since then., When our School first opened, Durning Hall was full morning, noon, and night, all year round. It was basically a community hub: open to all, all of the time. If you wanted an activity, you knew to go there first to see what they had on. Parents would put one child in my class, another in the Scouts group, and maybe even do an activity for themselves. Durning Hall served whole families! Aston-Mansfield seem to think that times have changed such that Forest Gate no longer needs central and affordable community space. That is certainly not my experience. Our classes are well-attended because we provide something that local children want and can afford to do due to our location and prices. Due to covid-, everyone seems to have a greater appreciation of the benefits of being more active (especially children) and how using a specific interest (like dance) keeps people engaged for longer as opposed to just running or going to the gym. Parents are happy that their children can walk/cycle to classes like ours (particularly with the new Low Traffic Neighbourhoods around Newham), and can even travel on their own and improve their independence., I object to the demolition and reduced space and use to the community. I don't object to the redevelopment of the community space as long as it is updated and used as a community space and building. A series of new flats 

In [43]:
# Now apply the seed model to the subset of text for Ealing 
topics_subset_seed, probs_subset_seed = topic_model_seed.transform(cleaned_ealing_text)

# Convert probabilities to NumPy array
probs_array_seed = np.array(probs_subset_seed)

Batches: 100%|██████████| 16/16 [00:02<00:00,  5.47it/s]
2025-05-27 10:31:06,505 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-05-27 10:31:07,846 - BERTopic - Dimensionality - Completed ✓
2025-05-27 10:31:07,847 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-05-27 10:31:07,858 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2025-05-27 10:31:07,902 - BERTopic - Probabilities - Completed ✓
2025-05-27 10:31:07,903 - BERTopic - Cluster - Completed ✓


In [44]:
# Identify topics with a probability > 0.5 in at least one entry
selected_topics = np.where(probs_array_seed > 0.5)[1]  
selected_topics = np.unique(selected_topics)  

# Find the main topic for each document
main_topics = np.argmax(probs_array_seed, axis=1)

# Count occurrences of each main topic
topic_counts = pd.Series(main_topics).value_counts().reset_index()
topic_counts.columns = ["Topic", "Count"]

# Get topic representations from BERTopic (full version)
topic_info = topic_model_seed.get_topic_info()
filtered_topics = topic_info[topic_info["Topic"].isin(selected_topics)][["Topic", "Representation"]]

# Merge with topic counts
final_topics = filtered_topics.merge(topic_counts, on="Topic", how="left").fillna(0)

# Get representative documents: Take 3 example documents for each topic
representative_docs = {}
for topic in selected_topics:
    doc_indices = np.where(main_topics == topic)[0]  # Get indices of docs assigned to this topic
    top_docs = [cleaned_ealing_text[i] for i in doc_indices[:3]]  # Get up to 3 docs
    representative_docs[topic] = top_docs

# Add representative documents to the dataframe
final_topics["Representative Docs"] = final_topics["Topic"].map(representative_docs)

# Display full data
pd.set_option("display.max_colwidth", None)  # Ensure full text is shown
final_topics

,Topic,Representation,Count,Representative Docs
0,0,"[community, space, for, to, that, hall, and, we, durning, in]",10,"[The area is heaving with flats and people and feels crowded, claustrophobic and concrete, I lived in for years and it is a tiny patch of land. With the already burgeoning towers across the road are you really hoping to turn into a sky scraper neighbourhood. You have around the corner with new tower blocks going up everywhere and with the largest tower block you can see for miles. Until you have more schools, doctor surgeries and hospitals to deal with all these people these should not be built. Greedy fat cat bosses should not be the priority over people livinv in the area., This proposal will negatively impact the local community. There will be inadequate parking and it will put a strain on access to facilities for the people nearby. It is also gross overdevelopment.]"
1,1,"[green, space, was, this, of, the, would, in, development, as]",56,"[In the shrubs next to the road you can find shrews, voles or something like this. Has a survey been done , what were the results and how do you plan to recreate this habitat you are destroying? Thanks, . Affecting ecology - Loss of mature trees and biodiversity. I note that the Bat report suggests that 'habitats supported on the site at provide no obvious roosting opportunities for bats'. I know this not to be true as our garden which backs onto the site has regular bat activity during twilight hours in the summer months. There are some beautiful trees on the site which not only aid the biodiversity of the area but are beneficial for the well bing of the residents. I believe that the council should be protecting these mature trees not taking them down., . Open space - the proposed increase in in residential properties, with resulting lack of open space is also of particular concern. In a post COVID world, planners should be creating developments with more open space not less.]"
2,2,"[school, nursery, next, children, construction, primary, and, noise, building, dust]",10,"[The comments made about Lucas and Townshend house being able to use the park instead of the garden they pay to maintain are also unacceptable., We live on nearby street. is already one of the most polluted streets in the area and a traffic bottleneck and the high street already overcrowded. Parking is already an issue . Such a big development will cause too much strain on all existing facilities, on parks and schools nearby and disrupt the local character. There is no provision for extra school places. Existing dentist & GP facilities are over capacity. No guarantee that these blocks will not go sky high like other developments. There will be an increase in noise and air pollution and more light will be blocked. It is not good for health. There is no consideration to blending in to surrounding architecture and the immediate and local urban landscape. In addition to the aesthetics of the proposed tower blocks and taking the height and position of these structures into account, neighbour's gardens in the locality will have sunlight hours reduced by approximately two hours per day, and a loss of privacy for too. The Press has already highlighted the over development of the boroughs of , and with regard to Electrical Grid supply to ., My principal objection would be that this development includes no provision for extra school places. Assuming flats, if half of those house say to . children each (that's a guess, but seems reasonable) then additional school places for to children might be needed in the local area. I'm a parent with a child in a local school, I'm concerned at the potential for overcrowding in local classes and the resulting impact on educational attainment.]"
3,3,"[height, high, street, the, area, of, design, is, keeping, buildings]",57,"[Yet another attempt by developers to push through a totally unsuitable project in this area. These tall skyscrapers are totally out of keeping with the area

## Try zero-shot topic modelling 

Another variation on BERTopic is to combine it with zero-shot topic modelling. Here I predefine a list of topics, and the model generates clusters around each one. 

In [45]:
# We define a number of topics that we know are in the documents
zeroshot_topic_list = ["Lost light sunlight view", 
                       "Loss of community", 
                       "Loss of greenspace", 
                       "Air pollution construction", 
                       "Noise pollution construction"]

In [46]:
# We fit our model using the zero-shot topics
# and we define a minimum similarity. For each document,
# if the similarity does not exceed that value, it will be used
# for clustering instead.
topic_model_zero = BERTopic(
    embedding_model=model,
    min_topic_size=15,
    zeroshot_topic_list=zeroshot_topic_list,
    zeroshot_min_similarity=.85,
    representation_model=KeyBERTInspired()
)

topics, probs = topic_model_zero.fit_transform(cleaned_place_text)

In [47]:
# Now apply the seed model to the subset of text for Ealing 
topics_subset_zero, probs_subset_zero = topic_model_zero.transform(cleaned_ealing_text)

# Convert probabilities to NumPy array
probs_array_zero = np.array(probs_subset_zero)

In [48]:
# Identify topics with a probability > 0.5 in at least one entry
selected_topics = np.where(probs_array_zero > 0.5)  
selected_topics = np.unique(selected_topics)  

# Find the main topic for each document
main_topics = np.argmax(probs_array_zero)

# Count occurrences of each main topic
topic_counts = pd.Series(main_topics).value_counts().reset_index()
topic_counts.columns = ["Topic", "Count"]

# Get topic representations from BERTopic (full version)
topic_info = topic_model_zero.get_topic_info()
filtered_topics = topic_info[topic_info["Topic"].isin(selected_topics)][["Topic", "Representation"]]

# Merge with topic counts
final_topics = filtered_topics.merge(topic_counts, on="Topic", how="left").fillna(0)

# Get representative documents: Take 3 example documents for each topic
representative_docs = {}
for topic in selected_topics:
    doc_indices = np.where(main_topics == topic)[0]  # Get indices of docs assigned to this topic
    top_docs = [cleaned_ealing_text[i] for i in doc_indices[:3]]  # Get up to 3 docs
    representative_docs[topic] = top_docs

# Add representative documents to the dataframe
final_topics["Representative Docs"] = final_topics["Topic"].map(representative_docs)

# Display full data
pd.set_option("display.max_colwidth", None)  # Ensure full text is shown
final_topics

,Topic,Representation,Count,Representative Docs
0,1,"[proposed, development, proposals, plans, proposal, application, concerns, consultation, planning, developers]",0.0,[]
1,3,"[pollution, noise, airport, traffic, mitigation, environmental, roads, polluted, nuisance, residential]",1.0,[Taking away trees .enough of highrise flats in the area.we need' privacy.Please consider local residence New '..Road leading to not acceptable.To much traffic.]
2,5,"[buildings, daylight, building, sunlight, curtains, sky, rooms, east, vertical, effect]",0.0,[]
3,6,"[buildings, storeys, neighbourhood, storey, building, housing, architecture, proposed, heights, tower]",0.0,[]
4,7,"[parking, park, bus, roadway, cars, building, car, permits, laneway, street]",0.0,[]
5,8,"[bus, facilities, dlr, transport, infrastructure, amenities, residents, overcrowded, commuting, routes]",0.0,[]
6,9,"[noise, acoustic, vibration, ambient, criteria, specifications, disturbance, residential, assessed, lpa]",0.0,[]
7,11,"[weekends, weekdays, hours, workers, weekend, dwellings, worker, disruption, shifts, work]",0.0,[]
8,12,"[buildings, residential, balcony, lock, building, apartment, security, construction, privacy, lighting]",0.0,[]
9,14,"[buildings, development, residential, construction, traffic, roads, pollution, residents, proposed, congestion]",0.0,[]


Note: the results of Zero-shot Topic Modelling suggests that this approach is too rigid. It atempts to find the topics identified, but then nothing is found. I think providing a seed_topic_list is a better way forward, as it weights the model towards favouring the seed topics, whilst still exploring and trying to find novel topics. 

## Different types of topic modelling: Latent Dirichlet Allocation

Latent dirichlet allocation uses bag of words to assign topics - so is not sensitive to the sequence of words, or the way sentences or comments are constructed. 

We start with some additional pre-processing, wherby we remove stop words (if, the, then...), remove all punctuation (.,:...), and also lemmatise (churches -> church, ...).

In [49]:
# use nltk to remove stopwords, punctuation and lemmatize the text
stop = set(stopwords.words('english'))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()

def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = ''.join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    return normalized

text_clean = [clean(doc).split() for doc in cleaned_place_text]

Looking at an example, can see how the original free text is reduced to a bag of descriptive words. 

In [50]:
text_clean[1578:1581]

[['noise', 'right', 'middle', 'development', 'next', 'pendant'],
 ['dangerous', 'work', 'going', 'next', 'school', 'nursery'],
 ['overstretching',
  'service',
  'existing',
  'resident',
  'already',
  'struggle',
  'receiving',
  'appropriate',
  'level',
  'service',
  'gym',
  'concierge']]

Convert the imput to a term-document matrix. This is a matrix which counts the occurrence of every term in each document and normalises the counts to create a matrix of values which can be used for LSA or LDA. 

In [51]:
# Create document-term matrix
dictionary = corpora.Dictionary(text_clean)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in text_clean] 

# Convert sparse to dense format
dense_matrix = [[tup[1] for tup in dictionary.doc2bow(doc)] for doc in text_clean]

In [52]:
# LDA model 
lda = LdaModel(doc_term_matrix, num_topics=10, id2word = dictionary, passes=50)

In [53]:
# print LDA topics as list 

lda_topics = lda.print_topics(num_topics=10, num_words=7)
lda_topics = [topic[1] for topic in lda_topics]
lda_topics = [topic.split('"') for topic in lda_topics]
lda_topics = [[word for word in topic if word.isalpha()] for topic in lda_topics]

for i, topic in enumerate(lda_topics):
    print(f"Topic {i}: {topic}")

Topic 0: ['community', 'space', 'would', 'year', 'building', 'child', 'school']
Topic 1: ['resident', 'system', 'pipe', 'noise', 'development', 'construction', 'proposed']
Topic 2: ['floor', 'th', 'ground', 'storage', 'turn', 'therefore', 'standard']
Topic 3: ['space', 'green', 'development', 'would', 'need', 'resident', 'flat']
Topic 4: ['noise', 'object', 'plant', 'development', 'residential', 'level', 'strongly']
Topic 5: ['air', 'quality', 'impact', 'pollution', 'area', 'emission', 'resident']
Topic 6: ['noise', 'school', 'construction', 'pollution', 'development', 'dust', 'nursery']
Topic 7: ['building', 'space', 'community', 'area', 'resident', 'already', 'new']
Topic 8: ['planning', 'building', 'development', 'area', 'high', 'application', 'use']
Topic 9: ['traffic', 'road', 'movement', 'meter', 'vehicle', 'increase', 'safety']
